# Video Generation Agent: Script-to-Video Pipeline

This notebook builds a multi-step agent pipeline that generates video scripts from a topic, breaks them into scenes, engineers optimized prompts for each scene, and produces videos using OpenAI's Sora model. The focus is on the **pipeline pattern** -- orchestrating multiple LLM calls and API interactions to transform a simple topic into production-ready video content.

## Architecture Overview

This is **not** just "send a prompt to Sora." It is a multi-step agent pipeline where each stage feeds into the next:

1. **Step 1: Script Generation** -- An LLM generates a structured video script with detailed scene descriptions, timing, camera movements, and transitions.
2. **Step 2: Scene Breakdown** -- The structured script is parsed into individual scenes with metadata for duration and visual style.
3. **Step 3: Prompt Engineering** -- Each scene description is transformed into an optimized Sora prompt that maximizes visual quality.
4. **Step 4: Video Generation** -- The Sora API generates video clips for each scene.

```
[Topic] --> [Script Writer] --> [Scene Breakdown] --> [Prompt Generator] --> [Sora API] --> [Video]
```

This pipeline pattern generalizes to any multi-modal content generation workflow: podcast production, slide deck creation, animated explainers, and more.

In [ ]:
# DEPENDENCY: pip install -q openai pydantic
# (packages should be pre-installed in venv before running this script)

In [ ]:
import os
import json
import time
from getpass import getpass
from pydantic import BaseModel, Field
from openai import OpenAI

# API Key Setup - uses environment variable or prompts for input
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")

client = OpenAI()
MODEL = "gpt-4o"


class Scene(BaseModel):
    """A single scene in the video script."""
    scene_number: int = Field(description="Scene sequence number")
    duration_seconds: int = Field(description="Duration of this scene in seconds")
    narration: str = Field(description="What is said or shown as text")
    visual_description: str = Field(description="Detailed description of what should be seen")
    camera_movement: str = Field(description="Camera movement, e.g. slow zoom in, pan left, static")
    transition: str = Field(description="Transition to next scene, e.g. fade, cut, dissolve")


class VideoScript(BaseModel):
    """Structured video script with scene breakdown."""
    title: str = Field(description="Video title")
    overall_mood: str = Field(description="The visual mood and tone")
    scenes: list[Scene] = Field(description="List of scenes in the video")

## Part 1: Script Generation

The first step in the pipeline is generating a **structured video script** from a topic. Rather than asking for a vague description, we use structured outputs to ensure the script contains all the metadata we need for downstream steps: scene-by-scene breakdowns with timing, camera movements, narration, and transitions.

This structured approach means each scene can be independently processed by the prompt optimizer and video generator.

In [ ]:
def generate_script(topic, duration_seconds=30, style="educational"):
    """Generate a structured video script using Pydantic models.
    
    Uses structured outputs with a Pydantic-derived JSON schema to ensure
    the script contains all necessary metadata for downstream processing.
    """
    response = client.responses.create(
        model=MODEL,
        input=f"""Create a video script for a {duration_seconds}-second {style} video about: {topic}

Guidelines:
- Break into 3-6 scenes that fill the total duration
- Visual descriptions should be vivid and specific
- Each scene should flow naturally into the next
- Consider the {style} style throughout""",
        text={
            "format": {
                "type": "json_schema",
                "name": "video_script",
                "strict": True,
                "schema": VideoScript.model_json_schema()
            }
        }
    )
    return VideoScript.model_validate_json(response.output_text)

In [ ]:
# Generate a script from a topic
script = generate_script(
    topic="The future of AI agents in everyday life",
    duration_seconds=30,
    style="cinematic educational"
)

print(f"Title: {script.title}")
print(f"Mood: {script.overall_mood}")
print(f"Scenes: {len(script.scenes)}")
print(f"Total duration: {sum(s.duration_seconds for s in script.scenes)}s")

for scene in script.scenes:
    print(f"\n  Scene {scene.scene_number} ({scene.duration_seconds}s)")
    print(f"  Visual: {scene.visual_description}")
    print(f"  Camera: {scene.camera_movement}")
    print(f"  Transition: {scene.transition}")

## Part 2: Sora Prompt Optimization

Sora prompts need to be optimized differently from standard text prompts. Effective Sora prompts are:

- **Visually specific** -- describe lighting, colors, textures, materials in detail
- **Cinematically aware** -- specify camera angles, movements, and lens effects
- **Mood-driven** -- convey the atmosphere and emotional tone
- **Concise but dense** -- pack maximum visual information into under 200 words

This step uses a separate LLM call to transform each scene's structured description into a prompt that Sora can render effectively. Think of it as a "prompt translator" -- converting narrative descriptions into visual generation instructions.

In [ ]:
def optimize_sora_prompt(scene, overall_mood):
    """Convert a Scene into an optimized Sora prompt.
    
    Sora responds best to prompts that focus on visual details,
    camera work, and atmosphere rather than narrative or dialogue.
    """
    response = client.responses.create(
        model=MODEL,
        input=f"""Convert this scene description into an optimized prompt for Sora (AI video generation).

Scene: {scene.model_dump_json()}
Overall Mood: {overall_mood}

Guidelines for Sora prompts:
- Be extremely specific about visual details (lighting, colors, textures)
- Describe camera movement explicitly
- Include the mood/atmosphere
- Mention the style (cinematic, documentary, animation, etc.)
- Keep it under 200 words
- Focus on what should be SEEN, not heard

Return just the optimized prompt text, nothing else."""
    )
    return response.output_text

In [ ]:
# Optimize prompts for all scenes
sora_prompts = []
for scene in script.scenes:
    prompt = optimize_sora_prompt(scene, script.overall_mood)
    sora_prompts.append({
        "scene_number": scene.scene_number,
        "duration": scene.duration_seconds,
        "prompt": prompt
    })
    print(f"Scene {scene.scene_number} prompt:")
    print(f"  {prompt[:200]}...")
    print()

## Part 3: Video Generation with Sora

Now we use the **OpenAI Video API** to generate video clips from our optimized prompts. The Sora API works asynchronously:

1. **Create** a video generation job with `client.videos.create()`
2. **Poll** for completion with `client.videos.retrieve()` (status goes from `queued` to `in_progress` to `completed`)
3. **Download** the finished MP4 with `client.videos.download_content()`

Available models:
- `sora-2` -- faster generation, good for iteration
- `sora-2-pro` -- higher quality output, slower

Parameters:
- `prompt` -- the text description of the video
- `seconds` -- clip duration: `4`, `8`, or `12`
- `size` -- resolution: `1280x720`, `720x1280`, `1792x1024`, or `1024x1792`

In [ ]:
def generate_video(prompt, duration_seconds=4, model="sora-2", size="1280x720"):
    """Generate a video using OpenAI's Sora model.
    
    Creates a video generation job, polls for completion, and
    returns the video object once finished.
    
    Args:
        prompt: Optimized text description for Sora.
        duration_seconds: Clip length -- must be 4, 8, or 12.
        model: 'sora-2' (fast) or 'sora-2-pro' (high quality).
        size: Resolution string, e.g. '1280x720'.
    
    Returns:
        The completed video object from the API.
    """
    # Sora only supports 4, 8, or 12 second clips
    allowed_durations = [4, 8, 12]
    sora_duration = min(allowed_durations, key=lambda x: abs(x - duration_seconds))
    
    print(f"  Creating video job (model={model}, {sora_duration}s, {size})...")
    
    # Step 1: Create the video generation job
    video = client.videos.create(
        model=model,
        prompt=prompt,
        seconds=str(sora_duration),
        size=size
    )
    
    video_id = video.id
    print(f"  Job created: {video_id}")
    
    # Step 2: Poll until the job completes
    while True:
        status = client.videos.retrieve(video_id)
        print(f"  Status: {status.status} (progress: {status.progress}%)")
        
        if status.status == "completed":
            print(f"  Video generation complete.")
            return status
        elif status.status == "failed":
            raise Exception(f"Video generation failed for job {video_id}")
        
        time.sleep(10)  # Poll every 10 seconds


def download_video(video_id, output_path):
    """Download a completed video to a local file.
    
    Args:
        video_id: The ID of the completed video.
        output_path: Local file path to save the MP4.
    """
    content = client.videos.download_content(video_id)
    with open(output_path, "wb") as f:
        f.write(content)
    print(f"  Saved: {output_path}")

In [ ]:
# Generate a video for the first scene as a demonstration
# NOTE: Sora API access requires an eligible OpenAI plan.
# Uncomment the lines below to run video generation.

first_scene = sora_prompts[0]
print(f"Generating video for Scene {first_scene['scene_number']}...")
print(f"Prompt: {first_scene['prompt'][:150]}...")
print()

# video = generate_video(
#     prompt=first_scene['prompt'],
#     duration_seconds=first_scene['duration'],
#     model="sora-2",
#     size="1280x720"
# )
# download_video(video.id, f"scene_{first_scene['scene_number']}.mp4")

print("(Video generation commented out -- uncomment to run with Sora API access)")

## Part 4: Full Pipeline

Now we combine all the steps into a single end-to-end pipeline function. Given just a topic, it:

1. Generates a structured script
2. Optimizes each scene into a Sora prompt
3. Generates videos for every scene

This is the core agent pattern: **decompose a high-level goal into specialized sub-tasks**, each handled by the best tool for the job.

In [ ]:
def generate_video_from_topic(topic, duration_seconds=30, style="educational", 
                               sora_model="sora-2", size="1280x720",
                               generate_videos=False):
    """Full pipeline: topic -> script -> prompts -> video(s).
    
    Args:
        topic: The subject for the video.
        duration_seconds: Total target duration of the video.
        style: Visual/narrative style (e.g., 'educational', 'cinematic').
        sora_model: 'sora-2' or 'sora-2-pro'.
        size: Video resolution string.
        generate_videos: Set to True to actually call the Sora API.
    
    Returns:
        Dictionary with script, prompts, and video results.
    """
    print(f"Generating video for: {topic}")
    print("=" * 60)
    
    # Step 1: Generate script
    print("\n[Step 1] Generating script...")
    script = generate_script(topic, duration_seconds, style)
    print(f"  Title: {script.title}")
    print(f"  Created {len(script.scenes)} scenes")
    
    # Step 2: Optimize prompts for each scene
    print("\n[Step 2] Optimizing Sora prompts...")
    prompts = []
    for scene in script.scenes:
        prompt = optimize_sora_prompt(scene, script.overall_mood)
        prompts.append({
            "scene_number": scene.scene_number,
            "duration": scene.duration_seconds,
            "prompt": prompt
        })
        print(f"  Scene {scene.scene_number}: prompt ready ({len(prompt)} chars)")
    
    # Step 3: Generate videos (optional -- requires Sora API access)
    print("\n[Step 3] Generating videos...")
    videos = []
    if generate_videos:
        for p in prompts:
            print(f"  Generating scene {p['scene_number']}...")
            video = generate_video(
                prompt=p['prompt'],
                duration_seconds=p['duration'],
                model=sora_model,
                size=size
            )
            videos.append(video)
            download_video(video.id, f"scene_{p['scene_number']}.mp4")
    else:
        print("  Skipped (set generate_videos=True to call Sora API)")
    
    print("\n" + "=" * 60)
    print("Pipeline complete.")
    
    return {
        "script": script,
        "prompts": prompts,
        "videos": videos
    }

In [ ]:
# Run the full pipeline (text steps only -- video generation is off by default)
result = generate_video_from_topic(
    topic="How AI agents are transforming software development",
    duration_seconds=30,
    style="modern tech documentary"
)

In [ ]:
# Inspect the generated prompts
for p in result['prompts']:
    print(f"Scene {p['scene_number']} ({p['duration']}s):")
    print(f"  {p['prompt']}")
    print()

## Part 5: Cost Estimation

Before running the full pipeline with video generation enabled, it is important to estimate costs. The pipeline has two cost components:

1. **LLM token costs** -- for script generation and prompt optimization (GPT-4o pricing)
2. **Video generation costs** -- Sora pricing per second of generated video

In [ ]:
def estimate_pipeline_cost(result):
    """Estimate the cost of a pipeline run.
    
    Provides rough estimates for LLM token usage and video generation.
    Actual costs depend on current OpenAI pricing.
    """
    # Rough token estimates (1 token ~ 4 characters)
    script = result['script']
    script_tokens = len(script.model_dump_json()) // 4
    prompt_tokens = sum(len(p['prompt']) // 4 for p in result['prompts'])
    total_llm_tokens = script_tokens + prompt_tokens
    
    # Video duration breakdown
    total_video_seconds = sum(p['duration'] for p in result['prompts'])
    num_scenes = len(result['prompts'])
    
    print("Cost Estimation")
    print("=" * 40)
    print(f"\nLLM Usage (GPT-4o):")
    print(f"  Script generation: ~{script_tokens} tokens")
    print(f"  Prompt optimization: ~{prompt_tokens} tokens")
    print(f"  Total LLM tokens: ~{total_llm_tokens} tokens")
    print(f"\nVideo Generation (Sora):")
    print(f"  Number of scenes: {num_scenes}")
    print(f"  Total video duration: {total_video_seconds}s")
    print(f"  Note: Sora pricing varies by model and resolution.")
    print(f"  Check https://openai.com/pricing for current rates.")


estimate_pipeline_cost(result)

## Summary

Key takeaways from this notebook:

- **Video generation benefits from a multi-step pipeline approach** -- a single prompt to Sora produces far weaker results than a structured script broken into optimized scene prompts.
- **Script generation provides structure and narrative flow** -- using structured outputs (JSON schema) ensures every scene has the metadata needed for downstream processing.
- **Prompt optimization improves Sora output quality significantly** -- translating narrative descriptions into visually specific, cinematically aware prompts is a distinct skill that an LLM can perform.
- **Each step uses the best model for the task** -- GPT-4o for reasoning and text generation, Sora for video generation.
- **Cost management matters** -- estimate total pipeline cost (LLM tokens + video generation) before running at scale.
- **This pattern generalizes** -- the same decompose-optimize-generate pipeline works for any multi-modal content workflow (podcasts, slide decks, animations, etc.).